# A2C with Robotic Simulations

Training a robot arm with Advantage Actor Critic method in Panda-Gym Environments using Stable-Baselines3 library.

**PandaReachDense-v3** agent is a robotic arm that is using the end-effector to target a green ball, no join control, in dense environment, i.e. a dense reward function provides a reward at each timestep.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/ColabNotebooks/DeepRL/actor_critics

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics


## Install dependencies

### Libraries for virtual display

Colab headless environment need to have a virtual screen to be able to render the environment and record frames, and generate a replay video

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip install pyvirtualdisplay

### Libraries for the robotic environment, dl learning, uploading and working with HF

In [ ]:
!pip install stable-baselines3[extra] # provides the brain that learns from the environment's state, takes action and process the reward to update the policy, implementations of RL algorithms in PyTorch
!pip install gymnasium                # toolkit and API for creating and interacting with virtual RL environments, high-level python class representing a Markov Decision Proces, able to start new episode, take action and visualize agent's current state
!pip install panda_gym                # provides a specialized set of robotic environments, featuring Franka Emika Panda robot - a 7 axes robot arm
!pip install huggingface_sb3
!pip install huggingface_hub

## Imports

In [ ]:
import os

import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub
                                                                                          # SB3 set of RL algorithms implemented in PyTorch
from stable_baselines3 import A2C                                                         # Advantage Actor Critic agent implemented with SB3
from stable_baselines3.common.evaluation import evaluate_policy                           # utility function for running a trained model on a given environment omputing the mean reward
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecVideoRecorder  # a moving average, normalizing wrapper for vectorized environment
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login
from huggingface_sb3 import package_to_hub
from huggingface_hub import upload_file


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
notebook_login()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# browse available gymansium environments
for i in gym.envs.registry.keys():
  print(i)

CartPole-v0
CartPole-v1
MountainCar-v0
MountainCarContinuous-v0
Pendulum-v1
Acrobot-v1
phys2d/CartPole-v0
phys2d/CartPole-v1
phys2d/Pendulum-v0
LunarLander-v3
LunarLanderContinuous-v3
BipedalWalker-v3
BipedalWalkerHardcore-v3
CarRacing-v3
Blackjack-v1
FrozenLake-v1
FrozenLake8x8-v1
CliffWalking-v1
CliffWalkingSlippery-v1
Taxi-v3
tabular/Blackjack-v0
tabular/CliffWalking-v0
Reacher-v2
Reacher-v4
Reacher-v5
Pusher-v2
Pusher-v4
Pusher-v5
InvertedPendulum-v2
InvertedPendulum-v4
InvertedPendulum-v5
InvertedDoublePendulum-v2
InvertedDoublePendulum-v4
InvertedDoublePendulum-v5
HalfCheetah-v2
HalfCheetah-v3
HalfCheetah-v4
HalfCheetah-v5
Hopper-v2
Hopper-v3
Hopper-v4
Hopper-v5
Swimmer-v2
Swimmer-v3
Swimmer-v4
Swimmer-v5
Walker2d-v2
Walker2d-v3
Walker2d-v4
Walker2d-v5
Ant-v2
Ant-v3
Ant-v4
Ant-v5
Humanoid-v2
Humanoid-v3
Humanoid-v4
Humanoid-v5
HumanoidStandup-v2
HumanoidStandup-v4
HumanoidStandup-v5
GymV21Environment-v0
GymV26Environment-v0
PandaReach-v3
PandaReachJoints-v3
PandaReachDense-v3
Pan

## **PandaReachDense-v3** Environment

Environments Wrapper Chain:

VecVideoRecorder<br>
>└─ VecNormalize<br>
>>└─ DummyVecEnv<br>
>>>└─ PandaReachDense-v3<br>

where DummyVecEnv converts panda-gym environment, based on PyBullet physics engine and gymansium to vectorised environment, VecNormalize normalizes input features, observation and the reward, VecVideoRecorder wraps a VecEnv object to record rendered image as mp4 video.

**VecNormalize** wrapper performce moving average normalization on ibservations and rewards, thus *Stabilizes Training* of RL algorithms which are sensitive to the scale and distribution of observations and rewards. Normalization scale of these values to consitent range, mean of zero and unit variance helps smooth the optimization process, handles *Different Scales* of observation variables like goal position, distance, position and velovity, ensureing that no single component dominates the learning process because of its values are larger. It also *Imporves Performance* byproviding the algorithm with more cosistent input data.

In [ ]:
env_id = "PandaReachDense-v3"

# create a single instance of the env
env = gym.make(env_id)

# by default env is intit to a random state, use seed with reset() if want the same state
# seed = 123
#env.reset(seed=seed)

# create multiple copies for training in parallel
env = make_vec_env(env_id, n_envs=4)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# get the state space and action space
s_size = env.observation_space.shape
a_size = env.action_space.shape

In [ ]:
print("_____ACTION SPACE_____ \n")
print("The Action Space is x, y, z movement: ", a_size)
print("Action Space sample", env.action_space.sample())      # take a random observation
print()
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is goal position, distance bw goal and current position, observation position and velocity: ", s_size)
print("Sample observation", env.observation_space.sample())  # take a random action

_____ACTION SPACE_____ 

The Action Space is x, y, z movement:  (3,)
Action Space sample [-0.26848772 -0.5989731   0.58601624]

_____OBSERVATION SPACE_____ 

The State Space is goal position, distance bw goal and current position, observation position and velocity:  None
Sample observation {'achieved_goal': array([ 8.958622 , -1.7443534,  4.7540083], dtype=float32), 'desired_goal': array([-5.4458585, -2.9188285,  1.8858304], dtype=float32), 'observation': array([ 6.277695 , -4.7510552, -1.0482299, -1.8355509,  3.3504357,
       -2.4989207], dtype=float32)}


In [ ]:
# an example of an individual observation array
# For VecEnv with Dict observation space, env.reset() returns a dictionary of arrays
observations = env.reset()
# accessing the observation array for the 'observation' key and then the observation for the first environment
print("observation array for the first environment:", observations['observation'][0])

# to see all observations for a specific key:
# print("observation array for all environments (observation key):", observations['observation'])
# to see the entire observation dictionary:
# print("entire observation dictionary:", observations)

observation array for the first environment: [ 8.5804430e-05 -4.9020774e-15  4.3262634e-04  0.0000000e+00
 -0.0000000e+00  0.0000000e+00]


## A2C Model

In [ ]:
model = A2C("MultiInputPolicy", env, verbose=1)

Using cuda device


## Train

In [ ]:
model.learn(500_000)

# save the model and VecNormalize objext assosiated with the env variable normalization statistics
model.save("a2c-PandaReachDense-v3")
env.save("vec_normalize.pkl")

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 43.4     |
|    ep_rew_mean        | -10.9    |
|    success_rate       | 0.222    |
| time/                 |          |
|    fps                | 303      |
|    iterations         | 100      |
|    time_elapsed       | 6        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | -4.25    |
|    explained_variance | 0.522    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -1.72    |
|    std                | 0.998    |
|    value_loss         | 0.289    |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 43.4     |
|    ep_rew_mean        | -12.1    |
|    success_rate       | 0.2      |
| time/                 |          |
|    fps                | 350      |
|    iterations         | 200      |
|

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 39.3     |
|    ep_rew_mean        | -9.99    |
|    success_rate       | 0.26     |
| time/                 |          |
|    fps                | 354      |
|    iterations         | 600      |
|    time_elapsed       | 33       |
|    total_timesteps    | 12000    |
| train/                |          |
|    entropy_loss       | -4.38    |
|    explained_variance | 0.955    |
|    learning_rate      | 0.0007   |
|    n_updates          | 599      |
|    policy_loss        | 0.0671   |
|    std                | 1.04     |
|    value_loss         | 0.0588   |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 40.2     |
|    ep_rew_mean        | -10.4    |
|    success_rate       | 0.23     |
| time/                 |          |
|    fps                | 354      |
|    iterations         | 700      |
|

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.97     |
|    ep_rew_mean        | -0.231   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 356      |
|    iterations         | 3100     |
|    time_elapsed       | 173      |
|    total_timesteps    | 62000    |
| train/                |          |
|    entropy_loss       | -3.78    |
|    explained_variance | 0.357    |
|    learning_rate      | 0.0007   |
|    n_updates          | 3099     |
|    policy_loss        | -0.102   |
|    std                | 0.854    |
|    value_loss         | 0.0015   |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 3.17     |
|    ep_rew_mean        | -0.255   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 357      |
|    iterations         | 3200     |
|

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.76     |
|    ep_rew_mean        | -0.213   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 364      |
|    iterations         | 5400     |
|    time_elapsed       | 296      |
|    total_timesteps    | 108000   |
| train/                |          |
|    entropy_loss       | -3.37    |
|    explained_variance | 0.548    |
|    learning_rate      | 0.0007   |
|    n_updates          | 5399     |
|    policy_loss        | -0.0351  |
|    std                | 0.744    |
|    value_loss         | 0.00148  |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.81     |
|    ep_rew_mean        | -0.216   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 365      |
|    iterations         | 5500     |
|

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.83     |
|    ep_rew_mean        | -0.232   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 366      |
|    iterations         | 5600     |
|    time_elapsed       | 305      |
|    total_timesteps    | 112000   |
| train/                |          |
|    entropy_loss       | -3.35    |
|    explained_variance | 0.46     |
|    learning_rate      | 0.0007   |
|    n_updates          | 5599     |
|    policy_loss        | 0.046    |
|    std                | 0.739    |
|    value_loss         | 0.00188  |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.77     |
|    ep_rew_mean        | -0.222   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 365      |
|    iterations         | 5700     |
|

## Continue Training

In [ ]:
# Define the environment ID
env_id = "PandaReachDense-v3"

# create a new 4-environment VecEnv for continued training
train_env = make_vec_env(env_id, n_envs=4)

# load the saved VecNormalize statistics from the original 4-environment training
# and apply them to the new 4-environment train_env
train_env = VecNormalize.load("vec_normalize.pkl", train_env)

# load the already trained agent and associate it with the new 4-environment train_env
model = A2C.load("a2c-PandaReachDense-v3.zip", env=train_env)

# Continue training the model on the 4-environment train_env
model.learn(500_000)

# Save the model after continued training
model.save("a2c-PandaReachDense-v3")

# save the VecNormalize statistics from the original 4-environment training (using the 'env' variable from earlier)
# which preserves the normalization stats based on the initial training data distribution.
train_env.save("vec_normalize.pkl")

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.85     |
|    ep_rew_mean        | -0.226   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 377      |
|    iterations         | 100      |
|    time_elapsed       | 5        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | 0.935    |
|    explained_variance | 0.917    |
|    learning_rate      | 0.0007   |
|    n_updates          | 175099   |
|    policy_loss        | -0.00843 |
|    std                | 0.187    |
|    value_loss         | 0.00022  |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.75     |
|    ep_rew_mean        | -0.212   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 394      |
|    iterations         | 200      |
|

## Evaluate

In [ ]:
# create an environment instance and wrap it in DummyVecEnv making it compatible with SB3 interface
eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
# load the normalization statistics saved during training and apply to evaluation environment
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# override the render mode which will allow the environment to render frames as RGB arrays, use for visualizations and video
eval_env.render_mode = "rgb_array"

# disable updating of normalization statistics during evaluation
eval_env.training = False
# disable the normalization of rewards during evaluation, allowing to see the actual rewards
eval_env.norm_reward = False

In [ ]:
model = A2C.load("a2c-PandaReachDense-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

Mean reward = -0.26 +/- 0.09


## Generate Video

In [ ]:
import os
import base64                                 # encoding binary data into ASCII characters

from pathlib import Path                      # to interact with file system paths
from IPython import display as ipythondisplay # displays varius output types including html

from stable_baselines3.common.vec_env.vec_video_recorder import VecVideoRecorder


When running into a non-graphical environment, "trick" graphical application into an available display

In [ ]:
os.system("Xvfb :1 -screen 0 1024x768x24 &")  # start a virutal display with shell command, a X virtual frame buffer
os.environ['DISPLAY'] = ':1'                  # in memory display saver that performs all graphical
                                              # operations in virtual memory without showing any output

In [ ]:
def record_video(env_id, model, video_length, prefix="", video_folder="videos/"):
    """
    Capture a video of the trained model performance in environment, by creating a
    DummyVecEnv for specified environment, rendering as RGB arrays, and wraping it
    with VecVideoRecorder, whcih handles the process of caputing

    :param env_id: (str)
    :param model: (RL model)
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    # start the video at step=0 and record video_length steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset() # to get the initial observation

    for _ in range(video_length):
        action, _ = model.predict(obs)
        obs, _, _, _ = eval_env.step(action)

    eval_env.close()      # close the video recorder

def show_videos(video_path="", prefix=""):
    """
    Display video from folder in a notebook
    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    # find all .mp4 files in specified path that match prefix
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        # the binary content of video file is read and encoded into base64 string in order to be embeded in HTML output
        video_b64 = base64.b64encode(mp4.read_bytes())
        # append an html tag, configured to autoplay, loop and display controls
        html.append(
            """<video alt="{}" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    # the list of html video tags are joined with line brake and displays the resulting html
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [ ]:
record_video("PandaReachDense-v3", model, video_length=500, prefix="a2c")

Saving video to /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-step-0-to-step-500.mp4
Moviepy - Building video /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-step-0-to-step-500.mp4.
Moviepy - Writing video /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-step-0-to-step-500.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-step-0-to-step-500.mp4


In [ ]:
show_videos("videos", prefix="a2c")

In [ ]:
#!apt-get install -y xvfb python-opengl > /dev/null 2>&1
#!xvfb-run -s "-screen 0 1400x900x24" <python_file>

## Publish

In [ ]:
notebook_login()

# configure Git to remember credentials
!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import gymnasium as gym
from huggingface_sb3 import package_to_hub
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecVideoRecorder

env_id = "PandaReachDense-v3"

# base environment for evaluation
eval_env_base = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])

# load VecNormalize statistics
try:
    eval_env_normalized = VecNormalize.load("vec_normalize.pkl", eval_env_base)
    eval_env_normalized.training = False
    eval_env_normalized.norm_reward = False
except FileNotFoundError:
    print("Normalization file 'vec_normalize.pkl' not found.")
    eval_env_normalized = eval_env_base

# wrap with VecVideoRecorder — must be the OUTERMOST wrapper
eval_env_for_hub = VecVideoRecorder(
    eval_env_normalized,
    video_folder="./videos",
    record_video_trigger=lambda step: step == 0,
    video_length=500,
    name_prefix=f"a2c-{env_id}-replay",
)

# call package_to_hub with the *outermost* env
package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env_for_hub,  # ✅ important
    repo_id=f"elliemci/a2c-{env_id}",
    commit_message="Commit with video replay",
)


ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saving video to /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-PandaReachDense-v3-replay-step-0-to-step-500.mp4
Moviepy - Building video /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-PandaReachDense-v3-replay-step-0-to-step-500.mp4.
Moviepy - Writing video /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-PandaReachDense-v3-replay-step-0-to-step-500.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-PandaReachDense-v3-replay-step-0-to-step-500.mp4
Saving video to /tmp/tmpkebr1vzm/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpkebr1vzm/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpkebr1vzm/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmpkebr1vzm/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo elliemci/a2c-PandaReachDense-v3 to the Hugging Face
Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

... (6 outputs left out)

  ...3ie907n/vec_normalize.pkl: 100%|##########| 2.81kB / 2.81kB            

  ...3ie907n/vec_normalize.pkl: 100%|##########| 2.81kB / 2.81kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/elliemci/a2c-PandaReachDense-v3/tree/main/


CommitInfo(commit_url='https://huggingface.co/elliemci/a2c-PandaReachDense-v3/commit/e074c21fec1991e3d7d06e40225ca12352f4db59', commit_message='Commit with video replay', commit_description='', oid='e074c21fec1991e3d7d06e40225ca12352f4db59', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/a2c-PandaReachDense-v3'), pr_revision=None, pr_num=None)

The Hugging Face package_to_hub does the following:<br>

1. Evaluates the model
2. Records a shotrt replay video
3. Uploads the model + assets<br>

In step 2 huggingface_sb3.utils.record_video  library assumes that eval_env.video_recorder exists directly on the environment being passed, but the environemnet has been wrapped and the
.video_record attribute lives inside the VecVideoRecorder wrappen, not on the inner DymmyVecEnv, thus HuggingFace automatic replay generator can't find /content/drive/.../a2c-PandaReachDense-v3-replay-step-0-to-step-500.mp4

## Manually Upload Video to the Hub Repo

In [ ]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="./videos/a2c-PandaReachDense-v3-replay-step-0-to-step-500.mp4",
    path_in_repo="replay.mp4",
    repo_id="elliemci/a2c-PandaReachDense-v3",
    repo_type="model",
)

# can update README.md
# <video controls autoplay loop src="https://huggingface.co/elliemci/a2c-PandaReachDense-v3/resolve/main/replay.mp4" width="400"></video>


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ay-step-0-to-step-500.mp4: 100%|##########|  330kB /  330kB            

CommitInfo(commit_url='https://huggingface.co/elliemci/a2c-PandaReachDense-v3/commit/ee9ebe0b8cc91e2f0dfae35bbcd176dcdbfae248', commit_message='Upload replay.mp4 with huggingface_hub', commit_description='', oid='ee9ebe0b8cc91e2f0dfae35bbcd176dcdbfae248', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/a2c-PandaReachDense-v3'), pr_revision=None, pr_num=None)

## **PandaPickAndPlace-v3** Environment

Steps for training another agent:

1. Define the environment
2. Vectorize the environments
3. Add wrapper to normalize the observations and rewards
4. Create A2C model
5. Train for 1M timesteps
6. Save model and VecNotmalize statistics
7. Evaluate agent
8. Generate a video
9. Publish trained model manually uploading the video


In [ ]:
env_id = "PandaPickAndPlace-v3"

# create a single instance of the env
env = gym.make(env_id)

# create multiple copies of the same env for training in parallel
env = make_vec_env(env_id, n_envs=4)
# a moving average normalizing wrapper performing normalization of observations and reward, with max absolute obsevation value of 10
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# create a A2C models with policy that handle the dict of different observational space inputs
model = A2C("MultiInputPolicy", env, verbose=1)

# train 2x500000 because of Colab limitiations
model.learn(500_000)
# continue training
#model.learn(500_000)

# save the model and VecNormalize objext assosiated with the env variable normalization statistics
model.save("a2c-PandaPickAndPlace-v3")
env.save("vec_ppap_normalize.pkl")

# evaluate the agent, loading saved normalized statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaPickAndPlace-v3")])
eval_env = VecNormalize.load("vec_ppap_normalize.pkl", eval_env)
eval_env.training = False
eval_env.norm_reward = False
mean_reward, std_reward = evaluate_policy(model, eval_env)
print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

# wrap with VecVideoRecorder — must be the OUTERMOST wrapper
eval_env_for_hub = VecVideoRecorder(
    eval_env,
    video_folder="./videos",
    record_video_trigger=lambda step: step == 0,
    video_length=500,
    name_prefix=f"a2c-{env_id}-replay",
)

package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env_for_hub,
    repo_id=f"elliemci/a2c-{env_id}",
    commit_message="First commit",
)

upload_file(
    path_or_fileobj="./videos/a2c-PandaPickAndPlace-v3-replay-step-0-to-step-500.mp4",
    path_in_repo="replay.mp4",
    repo_id="elliemci/a2c-PandaPickAndPlace-v3",
    repo_type="model",
)


Using cuda device
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 48.8     |
|    ep_rew_mean        | -48.8    |
|    success_rate       | 0.025    |
| time/                 |          |
|    fps                | 309      |
|    iterations         | 100      |
|    time_elapsed       | 6        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | -5.75    |
|    explained_variance | 0.934    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -0.699   |
|    std                | 1.02     |
|    value_loss         | 0.0322   |
------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 48.2     |
|    ep_rew_mean        | -48.1    |
|    success_rate       | 0.037    |
| time/                 |          |
|    fps                | 299      |
|    iterations         | 200      |
|    time_elapsed       | 13       |
|    total_timesteps    | 4000     |
| train/                |          |
|    entropy_loss       | -5.77    |
|    explained_variance | 0.952    |
|    learning_rate      | 0.0007   |
|    n_updates          | 199      |
|    policy_loss        | -0.473   |
|    std                | 1.02     |
|    value_loss         | 0.0288   |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 48.5     |
|    ep_rew_mean        | -48.5    |
|    success_rate       | 0.03     |
| time/                 |          |
|    fps                | 316      |
|    iterations         | 300      |
|

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Mean reward = -45.00 +/- 15.00
ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /content/drive/MyDrive/ColabNotebooks/DeepRL/actor_critics/videos/a2c-PandaPickAndPlace-v3-replay-step-0-to-step-500.mp4


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


... (12 outputs left out)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ay-step-0-to-step-500.mp4: 100%|##########|  492kB /  492kB            

CommitInfo(commit_url='https://huggingface.co/elliemci/a2c-PandaPickAndPlace-v3/commit/03f381c642cead7f76941e62efa0d220d4067ff5', commit_message='Upload replay.mp4 with huggingface_hub', commit_description='', oid='03f381c642cead7f76941e62efa0d220d4067ff5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/a2c-PandaPickAndPlace-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/a2c-PandaPickAndPlace-v3'), pr_revision=None, pr_num=None)

In [ ]:
# continue training
env_id = "PandaPickAndPlace-v3"
train_env = make_vec_env(env_id, n_envs=4)
# Load the correct normalization file for PandaPickAndPlace-v3
train_env = VecNormalize.load("vec_ppap_normalize.pkl", train_env)
model = A2C.load("a2c-PandaPickAndPlace-v3.zip", env=train_env)
model.learn(500_000)
model.save("a2c-PandaPickAndPlace-v3")
# Save the updated normalization statistics for PandaPickAndPlace-v3
train_env.save("vec_ppap_normalize.pkl")

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 50       |
|    ep_rew_mean        | -50      |
|    success_rate       | 0        |
| time/                 |          |
|    fps                | 379      |
|    iterations         | 100      |
|    time_elapsed       | 5        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | -1.81    |
|    explained_variance | 0.97     |
|    learning_rate      | 0.0007   |
|    n_updates          | 75099    |
|    policy_loss        | 0.00207  |
|    std                | 0.384    |
|    value_loss         | 2.89e-06 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 48.8     |
|    ep_rew_mean        | -48.8    |
|    success_rate       | 0.025    |
| time/                 |          |
|    fps                | 340      |
|    iterations         | 200      |
|

In [ ]:
# evaluate the agent, loading saved normalized statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaPickAndPlace-v3")])
eval_env = VecNormalize.load("vec_ppap_normalize.pkl", eval_env)
eval_env.training = False
eval_env.norm_reward = False

# Load the model trained on PandaPickAndPlace-v3
model = A2C.load("a2c-PandaPickAndPlace-v3.zip")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

Mean reward = -50.00 +/- 0.00


In [ ]:
# wrap with VecVideoRecorder — must be the OUTERMOST wrapper
eval_env_for_hub = VecVideoRecorder(
    eval_env,
    video_folder="./videos",
    record_video_trigger=lambda step: step == 0,
    video_length=500,
    name_prefix=f"a2c-{env_id}-replay",
)

package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env_for_hub,
    repo_id=f"elliemci/a2c-{env_id}",
    commit_message="First commit",
)